<a href="https://colab.research.google.com/github/RobinJames25/APP2-FSTACK/blob/main/AI_recruitement_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:

registered_ids = set()


# ==========================================
# 1. APPLICANT VALIDATION
# ==========================================

def check_requirements(applicant_id, name, education, years_of_experience, skills):

    permitted_skills = [
        "python",
        "javascript",
        "sql"
    ]

    if isinstance(skills, (list, tuple)):
        skills = list(skills)

    applicant_details = {
        "applicant_id": applicant_id,
        "name": name,
        "education": education,
        "years_of_experience": years_of_experience,
        "skills": skills,
        "processing_status": "pending_review",
        "missing_fields": [],
        "validation_errors": []
    }

    skills_are_valid_container = (
        skills is None or isinstance(skills, (list, tuple))
    )

    if not skills_are_valid_container:
        applicant_details["validation_errors"].append(
            "skills must be a list or tuple"
        )

    if not isinstance(applicant_details["applicant_id"], str):
        print("Applicant ID must be a string")
        applicant_details["processing_status"] = "invalid"
        return applicant_details

    applicant_details["applicant_id"] = (
        applicant_details["applicant_id"].strip()
    )

    if applicant_details["applicant_id"] == "":
        print("Applicant ID cannot be empty")
        applicant_details["processing_status"] = "invalid"
        return applicant_details

    if applicant_details["applicant_id"] in registered_ids:
        applicant_details["validation_errors"].append(
            "applicant_id has already been registered"
        )

    for qualification in ("name", "education", "years_of_experience", "skills"):
        details = applicant_details[qualification]

        if qualification in ("name", "education"):
            if details is None or (
                isinstance(details, str) and details.strip() == ""
            ):
                applicant_details["missing_fields"].append(qualification)

                if qualification == "name":
                    applicant_details["validation_errors"].append(
                        "name is required"
                    )

                continue

            if not isinstance(details, str):
                applicant_details["validation_errors"].append(
                    f"{qualification} must be a string"
                )
                continue

        if qualification == "years_of_experience":
            if details is None:
                applicant_details["missing_fields"].append(qualification)
                continue

            if (
                not isinstance(details, int)
                or isinstance(details, bool)
                or details < 0
            ):
                applicant_details["validation_errors"].append(
                    "years_of_experience must be a non-negative integer"
                )

        if qualification == "skills":
            if not skills_are_valid_container:
                continue

            if details is None or len(details) == 0:
                applicant_details["missing_fields"].append(qualification)
                continue

            for index, item in enumerate(details):
                if not isinstance(item, str) or item.strip() == "":
                    applicant_details["validation_errors"].append(
                        f"{qualification} contains an invalid item"
                    )
                    continue

                item = item.strip().lower()
                details[index] = item

                if item not in permitted_skills:
                    applicant_details["validation_errors"].append(
                        f"{item} is an invalid skill for this application"
                    )

    if applicant_details["validation_errors"]:
        applicant_details["processing_status"] = "invalid"
    elif applicant_details["missing_fields"]:
        applicant_details["processing_status"] = "pending_review"
    else:
        applicant_details["processing_status"] = "ready"

    registered_ids.add(applicant_details["applicant_id"])

    return applicant_details


# ==========================================
# 2. APPLICANT ELIGIBILITY
# ==========================================

def check_eligibility(applicant_details):

    eligibility = {
        "eligible": False,
        "status": "pending_review",
        "reasons": []
    }

    if not isinstance(applicant_details, dict):
        eligibility["reasons"].append(
            "Applicant details must be a dictionary"
        )
        return eligibility

    if "education" not in applicant_details:
      eligibility["reasons"].append("Education information is missing")

    if "years_of_experience" not in applicant_details:
      eligibility["reasons"].append("Years of experience is missing")

    if eligibility["reasons"]:
      return eligibility

    processing_status = applicant_details.get("processing_status")
    education = applicant_details.get("education")
    years = applicant_details.get("years_of_experience")

    permitted_education = [
        "bsc. software engineering",
        "bsc. financial engineering"
    ]

    if years is None:
        eligibility["reasons"].append(
            "Years of experience is missing"
        )
    elif not isinstance(years, int) or isinstance(years, bool):
      eligibility["reasons"].append(
          "Years of experience must be an interger"
      )
    elif years < 0:
      eligibility["reasons"].append(
          "Experience cannot be negative"
      )

    if not isinstance(education, str):
      eligibility["reasons"].append(
            "Education must be a string"
        )

    elif education.strip() == "":
       eligibility["reasons"].append(
        "Education information is missing"
    )

    if eligibility["reasons"]:
      return eligibility

    if processing_status != "ready":
        eligibility["reasons"].append(
            "Applicant has not passed validation"
        )
        return eligibility

    normalized_education = education.strip().lower()

    if normalized_education not in permitted_education:
        eligibility["status"] = "ineligible"
        eligibility["reasons"].append(
            "Applicant does not meet the education requirement"
        )

    if years < 2:
        eligibility["status"] = "ineligible"
        eligibility["reasons"].append(
            "Applicant does not meet the minimum experience requirement"
        )

    if eligibility["status"] != "ineligible":
        eligibility["eligible"] = True
        eligibility["status"] = "eligible"

    return eligibility


# ==========================================
# 3. APPLICANT SCORING
# ==========================================

def calculate_scores(qualification):

    if not isinstance(qualification, dict):
        print("Error: This function expects a dictionary")
        return None

    if qualification.get("processing_status") != "ready":
        print("Cannot calculate scores. Applicant is not ready.")
        return None

    candidate_score = 0
    years = qualification.get("years_of_experience")

    if (
        not isinstance(years, int)
        or isinstance(years, bool)
        or years < 0
    ):
        print("Invalid years of experience.")
        return None

    candidate_score += years * 5

    skills = qualification.get("skills")

    if not isinstance(skills, (list, tuple)):
        print("Invalid skills data.")
        return None

    for skill in skills:
        if not isinstance(skill, str) or skill.strip() == "":
            print("Invalid skill item.")
            return None

    normalized_skills = [
        skill.strip().lower() for skill in skills
    ]

    unique_skills = set(normalized_skills)

    for skill in unique_skills:
        if skill == "python":
            candidate_score += 15
        elif skill == "sql":
            candidate_score += 15
        elif skill == "javascript":
            candidate_score += 10

    return candidate_score


# ==========================================
# 4. MAIN APPLICATION FLOW
# ==========================================

if __name__ == "__main__":

    # qualification = check_requirements(
    #     "APP001",
    #     "Robins Obiri",
    #     123,
    #     None,
    #     ("Python", "SQL")
    # )

    # print("\nApplicant validation:")
    # print(qualification)

    # eligibility = check_eligibility(qualification)

    # print("\nApplicant eligibility:")
    # print(eligibility)

    # if eligibility["status"] == "eligible":

    #     score = calculate_scores(qualification)

    #     print("\nApplicant score:")
    #     print(score)

    # else:
    #     print("\nApplicant cannot proceed to scoring.")

    qualification = {
        "processing_status": "ready",
        "education": "Bsc. Accounting",
        "years__experience": 1
    }

    result = check_eligibility(qualification)
    print(result)


{'eligible': False, 'status': 'pending_review', 'reasons': ['Years of experience is missing']}
